# CO2 Emissions & Life Satisfaction

### Dependencies

In [1]:
# !pip install plotly openpyxl scikit-learn pandas numpy dash json

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

### Data Loading & Cleaning
DISCLAIMER: Lowkey used Claude Code for this part... I'm NOT manually remapping the country names between the datasets that don't fully match lol.
* see comments for code that is generated. Proof read?

In [3]:
whr_raw = pd.read_excel('WHR25_Data_Figure_2.1v3.xlsx')
whr_raw.columns = [c.strip() if isinstance(c, str) else c for c in whr_raw.columns]

# AI generated
KEEP_COLS = [
    'Year', 'Country name', 'Life evaluation (3-year average)',
    'Explained by: Log GDP per capita', 'Explained by: Social support',
    'Explained by: Healthy life expectancy',
    'Explained by: Freedom to make life choices',
    'Explained by: Generosity', 'Explained by: Perceptions of corruption',
]
# -----------

whr = whr_raw[KEEP_COLS].dropna(subset=['Year', 'Country name']).copy()
whr['Year'] = whr['Year'].astype(int)
whr = whr[(whr['Year'] >= 2014) & (whr['Year'] <= 2024)]

whr.head()

,Year,Country name,Life evaluation (3-year average),Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption
0,2024,Afghanistan,1.364,0.649,0.0,0.155,0.0,0.075,0.135
1,2023,Afghanistan,1.721,0.628,0.0,0.242,0.0,0.091,0.088
2,2022,Afghanistan,1.859,0.645,0.0,0.087,0.0,0.093,0.059
3,2021,Afghanistan,2.404,0.758,0.0,0.289,0.0,0.089,0.005
4,2020,Afghanistan,2.523,0.370,0.0,0.126,0.0,0.122,0.010


In [4]:
co2_raw = pd.read_csv('co2-emissions-per-capita.csv')
co2_raw.columns = ['country', 'iso_code', 'Year', 'co2_per_capita']

# Drop aggregate rows -> they have no ISO-3 code
co2 = co2_raw.dropna(subset=['iso_code']).copy()
co2['iso_code'] = co2['iso_code'].str.strip()
co2 = co2[(co2['Year'] >= 2014) & (co2['Year'] <= 2024)]

co2.head()

,country,iso_code,Year,co2_per_capita
65,Afghanistan,AFG,2014,0.265233
66,Afghanistan,AFG,2015,0.277384
67,Afghanistan,AFG,2016,0.248005
68,Afghanistan,AFG,2017,0.260895
69,Afghanistan,AFG,2018,0.277372


In [5]:
# AI generated

# 15 WHR names don't match CO2 entity names directly.
# 3 entities (North Cyprus, Puerto Rico, Somaliland Region) have no CO2 counterpart and are excluded from the merge.

NAME_MAP = {
    "Côte d'Ivoire"         : "Cote d'Ivoire",
    'DR Congo'               : 'Democratic Republic of Congo',
    'Hong Kong SAR of China' : 'Hong Kong',
    'Lao PDR'                : 'Laos',
    'Republic of Korea'      : 'South Korea',
    'Republic of Moldova'    : 'Moldova',
    'Russian Federation'     : 'Russia',
    'State of Palestine'     : 'Palestine',
    'Swaziland'              : 'Eswatini',
    'Taiwan Province of China': 'Taiwan',
    'Türkiye'                : 'Turkey',
    'Viet Nam'               : 'Vietnam',
    'North Cyprus'           : None,        # no CO2 data
    'Puerto Rico'            : None,        # no CO2 data
    'Somaliland Region'      : None,        # no CO2 data
}
# ----------- 

whr['co2_key'] = whr['Country name'].map(lambda x: NAME_MAP.get(x, x))
whr_clean = whr[whr['co2_key'].notna()].copy()

# merge on harmonised name + year
merged = whr_clean.merge(
    co2[['country', 'iso_code', 'Year', 'co2_per_capita']],
    left_on=['co2_key', 'Year'],
    right_on=['country', 'Year'],
    how='inner',
).drop(columns=['country'])

merged = merged.rename(columns={'Life evaluation (3-year average)': 'life_satisfaction'})

merged.head()

,Year,Country name,life_satisfaction,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,co2_key,iso_code,co2_per_capita
0,2024,Afghanistan,1.364,0.649,0.0,0.155,0.0,0.075,0.135,Afghanistan,AFG,0.253848
1,2023,Afghanistan,1.721,0.628,0.0,0.242,0.0,0.091,0.088,Afghanistan,AFG,0.253682
2,2022,Afghanistan,1.859,0.645,0.0,0.087,0.0,0.093,0.059,Afghanistan,AFG,0.250620
3,2021,Afghanistan,2.404,0.758,0.0,0.289,0.0,0.089,0.005,Afghanistan,AFG,0.246719
4,2020,Afghanistan,2.523,0.370,0.0,0.126,0.0,0.122,0.010,Afghanistan,AFG,0.284590


In [6]:
# AI generated

# ── World-region mapping ─────────────────────────────────────────────────────
REGION_MAP = {
    # Africa
    **dict.fromkeys(['Algeria','Angola','Benin','Botswana','Burkina Faso','Burundi',
        'Cameroon','Central African Republic','Chad','Comoros','Congo',
        'Democratic Republic of Congo',"Cote d'Ivoire",'Djibouti','Egypt',
        'Ethiopia','Eswatini','Gabon','Gambia','Ghana','Guinea','Kenya','Lesotho',
        'Liberia','Libya','Madagascar','Malawi','Mali','Mauritania','Mauritius',
        'Morocco','Mozambique','Namibia','Niger','Nigeria','Rwanda','Senegal',
        'Sierra Leone','Somalia','South Africa','South Sudan','Sudan',
        'Tanzania','Togo','Tunisia','Uganda','Zambia','Zimbabwe'], 'Africa'),
    # Americas
    **dict.fromkeys(['Argentina','Belize','Bolivia','Brazil','Canada','Chile',
        'Colombia','Costa Rica','Cuba','Dominican Republic','Ecuador','El Salvador',
        'Guatemala','Guyana','Haiti','Honduras','Jamaica','Mexico','Nicaragua',
        'Panama','Paraguay','Peru','Suriname','Trinidad and Tobago',
        'United States','Uruguay','Venezuela'], 'Americas'),
    # Asia-Pacific
    **dict.fromkeys(['Afghanistan','Australia','Bangladesh','Bhutan','Cambodia',
        'China','Hong Kong','India','Indonesia','Japan','Kazakhstan','Kyrgyzstan',
        'Laos','Malaysia','Maldives','Mongolia','Myanmar','Nepal','New Zealand',
        'Pakistan','Philippines','South Korea','Singapore','Sri Lanka','Taiwan',
        'Tajikistan','Thailand','Turkmenistan','Uzbekistan','Vietnam'], 'Asia-Pacific'),
    # Europe
    **dict.fromkeys(['Albania','Armenia','Austria','Azerbaijan','Belarus','Belgium',
        'Bosnia and Herzegovina','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
        'Estonia','Finland','France','Georgia','Germany','Greece','Hungary',
        'Iceland','Ireland','Israel','Italy','Kosovo','Latvia','Lithuania',
        'Luxembourg','Malta','Moldova','Montenegro','Netherlands','North Macedonia',
        'Norway','Poland','Portugal','Romania','Russia','Serbia','Slovakia',
        'Slovenia','Spain','Sweden','Switzerland','Turkey','Ukraine',
        'United Kingdom'], 'Europe'),
    # Middle East
    **dict.fromkeys(['Bahrain','Iran','Iraq','Jordan','Kuwait','Lebanon',
        'Oman','Qatar','Saudi Arabia','Palestine','Syria',
        'United Arab Emirates','Yemen'], 'Middle East'),
}

# -----------

merged['region'] = merged['co2_key'].map(REGION_MAP).fillna('Other')
print(merged['region'].value_counts())

region
Europe          502
Africa          451
Asia-Pacific    311
Americas        251
Middle East     120
Name: count, dtype: int64


#### Merge quality check
I don't trust that mf Claude

In [7]:
print(f"Final dataset: {merged.shape[0]:,} rows, {merged['Country name'].nunique()} countries")
print(merged['region'].value_counts())
merged[['co2_per_capita','life_satisfaction']].describe()

Final dataset: 1,635 rows, 162 countries
region
Europe          502
Africa          451
Asia-Pacific    311
Americas        251
Middle East     120
Name: count, dtype: int64


,co2_per_capita,life_satisfaction
count,1635.000000,1635.000000
mean,4.616553,5.463914
std,5.375596,1.130618
min,0.028531,1.364000
25%,0.865592,4.599500
50%,3.117392,5.504700
75%,6.337359,6.313000
max,41.310200,7.842000


### Visual 1: CO2 per Capita vs. Life Satisfaction
**Argument:** The Development Paradox (positive correlation) + Nonlinear cubic relationship. 

In [8]:
# cubic reg on the full dataset (all years pooled) -> see original EDA
X_co2 = merged[['co2_per_capita']].values
y_sat  = merged['life_satisfaction'].values

poly3 = PolynomialFeatures(degree=3, include_bias=True)
X_cubic = poly3.fit_transform(X_co2)
cubic_model = LinearRegression().fit(X_cubic, y_sat)
r2_cub = r2_score(y_sat, cubic_model.predict(X_cubic))
pearson_r = np.corrcoef(X_co2.ravel(), y_sat)[0, 1]

# log-spaced points so the line is dense at low CO2
x_smooth = np.exp(np.linspace(np.log(0.03), np.log(42), 400))
y_smooth = cubic_model.predict(poly3.transform(x_smooth.reshape(-1, 1)))

print(f"Pearson r  = {pearson_r:.3f}")
print(f"Cubic R^2   = {r2_cub:.3f}")

Pearson r  = 0.519
Cubic R^2   = 0.475


In [9]:
# colors
REGION_COLORS = {
    'Africa':"#11A5C0",             # teal
    'Americas':'#E8734A',           # coral
    'Asia-Pacific':'#F5A623',       # amber
    'Europe':'#7C3AED',             # violet
    'Middle East':'#2D8B4E',        # green
    'Other':'#9CA3AF',              # gray
}

plot_df = merged.sort_values(['Year', 'Country name']).copy()

# base scatter plot
fig1 = px.scatter(
    plot_df,
    x='co2_per_capita',
    y='life_satisfaction',
    animation_frame='Year',
    animation_group='Country name',
    color='region',
    color_discrete_map=REGION_COLORS,
    hover_name='Country name',
    hover_data={
        'co2_per_capita':':.2f',
        'life_satisfaction':':.2f',
        'region':True,
        'Year':False,
        'co2_key':False,
    },
    labels={
        'co2_per_capita':'CO2 per Capita (tonnes/person)',
        'life_satisfaction':'Life Satisfaction (0–10)',
        'region':'Region',
    },
    title='CO2 Emissions vs. Life Satisfaction (2014–2024)',
    log_x=True,
    range_x=[0.025, 50],
    range_y=[0.8, 9],
)

# markers
fig1.update_traces(
    marker=dict(size=9, opacity=0.78, line=dict(width=0.6, color='white')),
    selector=dict(mode='markers'),
)

# trace reg line
reg_trace = go.Scatter(
    x=x_smooth, y=y_smooth,
    mode='lines',
    name='Cubic fit (all years)',
    line=dict(color='#2E3B4E', width=2.5, dash='dot'),
    showlegend=True,
    hovertemplate='CO₂ = %{x:.2f}<br>Predicted LS = %{y:.2f}<extra>Cubic fit</extra>',
)
fig1.add_trace(reg_trace)

# repeat reg line so it's there for every animation frame
for frame in fig1.frames:
    frame.data = tuple(frame.data) + (
        go.Scatter(
            x=x_smooth, y=y_smooth,
            mode='lines',
            name='Cubic fit',
            line=dict(color='#2E3B4E', width=2.5, dash='dot'),
            showlegend=False,
        ),
    )

# annotations
annotations = [
    # stats box
    dict(
        text=(f'<b>Pearson r = {pearson_r:.2f}</b><br>'
              f'Cubic R² = {r2_cub:.2f}'),
        xref='paper', yref='paper', x=0.02, y=0.98,
        showarrow=False, align='left',
        bgcolor='rgba(255,255,255,0.88)',
        bordercolor='#CCCCCC', borderwidth=1,
        font=dict(size=11, family='Arial'),
    ),
    # industrialisation lab
    dict(
        text='← Industrialisation<br>   phase',
        xref='paper', yref='paper', x=0.09, y=0.1,
        showarrow=False, font=dict(size=10, color='#555555'),
        align='center',
    ),
    # diminishing returns
    dict(
        text='Diminishing returns →',
        xref='paper', yref='paper', x=0.72, y=1,
        showarrow=False, font=dict(size=10, color='#555555'),
    ),
]

# plot layout
fig1.update_layout(
    xaxis_title='CO2 per Capita: log scale (tonnes / person)',
    yaxis_title='Life Satisfaction (Cantril Ladder, 0–10)',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#2E3B4E'),
    legend=dict(
        title=dict(text='<b>Region</b>', font=dict(size=12)),
        x=1.01, y=1, xanchor='left',
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#E0E0E0', borderwidth=1,
    ),
    annotations=annotations,
    height=580, width=960,
    margin=dict(l=70, r=180, t=70, b=70),
)
fig1.update_xaxes(showgrid=True, gridcolor='#F2F2F2', zeroline=False,
                  tickfont=dict(size=11))
fig1.update_yaxes(showgrid=True, gridcolor='#F2F2F2', zeroline=False,
                  tickfont=dict(size=11))

# animation speed
fig1.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 900
fig1.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

fig1.show()

Might wanna add another annoation regarding that sudden upward curve of the cubic fit -> it's hard to fully capture diminishing returns mathematically. It got the downward curve right, but it spikes up again. 
* Quatar & Trinidad are the best examples of points located within that diminishing returns zone (high emissions, lower satisfaction comapred to points before them)

## Visual 2: Life Satisfaction & CO2 per Capita
**Argument:** Geographic and regional structural inequality

In [10]:
years = sorted(merged['Year'].unique())

# per year lookup dict
def get_year_df(yr):
    """Return a clean sub-frame for one year, one row per country."""
    g = (merged[merged['Year'] == yr]
         .groupby('iso_code', as_index=False)
         .agg(life_satisfaction=('life_satisfaction','mean'),
              co2_per_capita=('co2_per_capita','mean'),
              country_name=('Country name','first')))
    g['co2_log'] = np.log1p(g['co2_per_capita'])   # log-compress for color scale
    return g

# fig with the two geo subplots
fig2 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'choropleth'}, {'type': 'choropleth'}]],
    subplot_titles=[
        '<b>Life Satisfaction Score</b> (0–10)',
        '<b>CO2 per Capita</b> (log-compressed scale)'
    ],
    horizontal_spacing=0.04,
)

init = get_year_df(years[0])

# left map: Life Satisfaction (diverging: red-yellow-green centered at 5.5ish. Idk scale is subjective might change)
fig2.add_trace(
    go.Choropleth(
        locations=init['iso_code'], z=init['life_satisfaction'],
        text=init['country_name'],
        colorscale='RdYlGn',
        zmin=1.0, zmid=5.5, zmax=8.5,
        colorbar=dict(
            title='Score', x=0.455, len=0.72, thickness=14,
            tickvals=[1,3,5.5,7,8.5],
            ticktext=['1','3','5.5 ★','7','8.5'],
        ),
        hovertemplate='<b>%{text}</b><br>Life Satisfaction: %{z:.2f}<extra></extra>',
        showscale=True,
        geo='geo',
    ),
    row=1, col=1,
)

# right map: CO2 (sequential blues; log-compressed so gulf states don't dominate)
CO2_MAX_LOG = np.log1p(42)   # 3.76 log scale ceiling (big math)
fig2.add_trace(
    go.Choropleth(
        locations=init['iso_code'], z=init['co2_log'],
        text=init['country_name'],
        customdata=init['co2_per_capita'],
        colorscale='Blues',
        zmin=0, zmax=CO2_MAX_LOG,
        colorbar=dict(
            title='tonnes/person', x=1.00, len=0.72, thickness=14,
            tickvals=[np.log1p(v) for v in [0, 1, 3, 8, 20, 42]],
            ticktext=['0','1','3','8','20','42'],
        ),
        hovertemplate='<b>%{text}</b><br>CO2: %{customdata:.2f} tonnes/person<extra></extra>',
        showscale=True,
        geo='geo2',
    ),
    row=1, col=2,
)

# animation frame (1 frame = 1 year)
frames = []
for yr in years:
    yrdf = get_year_df(yr)
    frames.append(go.Frame(
        data=[
            go.Choropleth(
                locations=yrdf['iso_code'], z=yrdf['life_satisfaction'],
                text=yrdf['country_name'],
                colorscale='RdYlGn', zmin=1.0, zmid=5.5, zmax=8.5,
            ),
            go.Choropleth(
                locations=yrdf['iso_code'], z=yrdf['co2_log'],
                text=yrdf['country_name'],
                customdata=yrdf['co2_per_capita'],
                colorscale='Blues', zmin=0, zmax=CO2_MAX_LOG,
            ),
        ],
        name=str(yr),
    ))
fig2.frames = frames

# slider
slider_steps = [
    dict(
        args=[[str(yr)],
              {'frame': {'duration': 700, 'redraw': True},
               'mode': 'immediate',
               'transition': {'duration': 300}}],
        label=str(yr), method='animate',
    )
    for yr in years
]

# play/pause btns
play_buttons = dict(
    type='buttons', showactive=False,
    y=-0.08, x=0.06, xanchor='right', yanchor='top',
    buttons=[
        dict(label='▶ Play',
             method='animate',
             args=[None, {'frame': {'duration': 900, 'redraw': True},
                          'fromcurrent': True,
                          'transition': {'duration': 400}}]),
        dict(label='⏸ Pause',
             method='animate',
             args=[[None], {'frame': {'duration': 0, 'redraw': False},
                            'mode': 'immediate'}]),
    ],
)

# layout
fig2.update_layout(
    title=dict(
        text='Life Satisfaction & CO2 Emissions Across 162 Countries  (2014–2024)',
        font=dict(size=15, family='Arial'),
    ),
    height=480, width=1200,
    paper_bgcolor='white',
    font=dict(family='Arial', size=11, color='#2E3B4E'),
    margin=dict(l=10, r=10, t=80, b=120),
    sliders=[dict(
        steps=slider_steps,
        active=0,
        x=0.08, y=-0.04, len=0.85,
        pad={'t': 50},
        currentvalue=dict(
            prefix='Year: ', visible=True, xanchor='center',
            font=dict(size=14, color='#2E3B4E'),
        ),
        transition=dict(duration=300),
    )],
    updatemenus=[play_buttons],
    annotations=[
        dict(
            text='← Lowest satisfaction: Sub-Saharan Africa & South Asia',
            xref='paper', yref='paper', x=0.2, y=-0.2,
            showarrow=False, font=dict(size=10, color='#666'),
        ),
        dict(
            text='Highest emitters: Gulf States, North America, Australia →',
            xref='paper', yref='paper', x=0.93, y=-0.22,
            showarrow=False, xanchor='right', font=dict(size=10, color='#666'),
        ),
    ],
)

# style both geo projections the same
geo_style = dict(
    showcoastlines=True, coastlinecolor='#C0C0C0',
    showland=True, landcolor='#F7F7F7',
    showocean=True, oceancolor='#E6F2F8',
    showframe=False,
    projection_type='natural earth',
    showlakes=False,
)
fig2.update_geos(geo_style, 'geo')
fig2.update_geos(geo_style, 'geo2')

fig2.show()

#### This is broken. I'll keep trying to fix it.

## Visual 3: R² and Regression Coefficients
**Argument:** CO2 is a proxy for development, not an independent driver (Points 2 & 3 of our design doc).  

In [11]:
# prep modelling data
FEATURES = [
    'Explained by: Log GDP per capita',
    'Explained by: Social support',
    'Explained by: Healthy life expectancy',
    'Explained by: Freedom to make life choices',
    'Explained by: Generosity',
    'Explained by: Perceptions of corruption',
    'co2_per_capita',
]
FEATURE_LABELS = [
    'Log GDP per capita',
    'Social support',
    'Healthy life expectancy',
    'Freedom of choice',
    'Generosity',
    'Perceptions of corruption',
    'CO2 per capita',
]

model_df = merged[FEATURES + ['life_satisfaction']].dropna().copy()
X_all  = model_df[FEATURES].values
y_all  = model_df['life_satisfaction'].values
X_co2_ = model_df[['co2_per_capita']].values

# CO2-only polynomial models
def fit_poly(deg, X, y):
    pf = PolynomialFeatures(degree=deg)
    Xp = pf.fit_transform(X)
    m  = LinearRegression().fit(Xp, y)
    return r2_score(y, m.predict(Xp))

r2_lin  = fit_poly(1, X_co2_, y_all)
r2_quad = fit_poly(2, X_co2_, y_all)
r2_cub  = fit_poly(3, X_co2_, y_all)

# standardised multiple reg 
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
multi_m  = LinearRegression().fit(X_scaled, y_all)
r2_multi = r2_score(y_all, multi_m.predict(X_scaled))
coefs    = multi_m.coef_

# bootstrap 95% CIs (1,000 resamples)
rng = np.random.default_rng(42)
boot_coefs = np.array([
    LinearRegression().fit(
        X_scaled[idx := rng.choice(len(X_scaled), len(X_scaled))], y_all[idx]
    ).coef_
    for _ in range(1000)
])
ci_lo = np.percentile(boot_coefs, 2.5,  axis=0)
ci_hi = np.percentile(boot_coefs, 97.5, axis=0)

print("R^2 scores")
for name, val in zip(['Linear','Quadratic','Cubic','Multiple'],
                     [r2_lin, r2_quad, r2_cub, r2_multi]):
    print(f"  {name:<15}: {val:.3f}")
print("\nStandardised coefficients:")
for lab, c, lo, hi in zip(FEATURE_LABELS, coefs, ci_lo, ci_hi):
    print(f"  {lab:<38}: {c:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

R^2 scores
  Linear         : 0.281
  Quadratic      : 0.439
  Cubic          : 0.481
  Multiple       : 0.722

Standardised coefficients:
  Log GDP per capita                    : +0.274  95% CI [+0.211, +0.338]
  Social support                        : +0.277  95% CI [+0.213, +0.343]
  Healthy life expectancy               : +0.334  95% CI [+0.284, +0.383]
  Freedom of choice                     : +0.190  95% CI [+0.138, +0.245]
  Generosity                            : +0.115  95% CI [+0.070, +0.162]
  Perceptions of corruption             : +0.113  95% CI [+0.064, +0.163]
  CO2 per capita                        : +0.105  95% CI [+0.067, +0.146]


In [12]:
import copy
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# colors
TEAL_GRAD = ['#A8D5CF', '#1A9C87', '#155F56']
CORAL = '#E8734A'
TEAL_POS = '#1A9C87'
SLATE = '#2E3B4E'
INACTIVE_COLOR = '#CCCCCC'

r2_vals = [r2_lin, r2_quad, r2_cub, r2_multi]
bar_colors_left = TEAL_GRAD + [CORAL]
bar_labels_left = ['Linear (CO2 only)', 'Quadratic (CO2 only)',
                    'Cubic (CO2 only)', 'Multiple Regression']
bar_colors_right = [TEAL_POS if abs(c) > 0.05 else CORAL for c in coefs]

# figure
fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>R² by Model</b>',
        '<b>Standardised Regression Coefficients</b><br>'
        '<span style="font-size:11px">'
    ],
    column_widths=[0.40, 0.60],
    horizontal_spacing=0.14,
)

# Left panel: one trace per bar
for label, val, color in zip(bar_labels_left, r2_vals, bar_colors_left):
    fig3.add_trace(
        go.Bar(
            name=label,
            x=[label],
            y=[val],
            marker_color=color,
            marker_line=dict(color='white', width=1.5),
            text=[f'{val:.2f}'],
            textposition='outside',
            textfont=dict(size=12, color=SLATE),
            width=0.55,
            legendgroup=label,
            hovertemplate=f'<b>{label}</b><br>R² = {val:.3f}<extra></extra>',
        ),
        row=1, col=1,
    )

fig3.add_hline(
    y=r2_multi, line_dash='dash', line_color=CORAL, line_width=1.8,
    row=1, col=1,
)
fig3.add_annotation(
    xref='x', yref='y',
    x=1.5, y=r2_multi + 0.05,
    text=f'Multiple R² = {r2_multi:.2f}',
    showarrow=False,
    font=dict(size=10, color=CORAL, family='Arial'),
    row=1, col=1,
)

# Right panel: one trace per feature
for i, (label, c, lo, hi, color) in enumerate(
        zip(FEATURE_LABELS, coefs, ci_lo, ci_hi, bar_colors_right)):
    fig3.add_trace(
        go.Bar(
            name=f'feat_{i}',
            x=[c],
            y=[label],
            orientation='h',
            marker_color=color,
            marker_line=dict(color='white', width=0.8),
            error_x=dict(
                type='data', symmetric=False,
                array=[hi - c],
                arrayminus=[c - lo],
                color='#666', thickness=1.8, width=5,
            ),
            showlegend=False,
        ),
        row=1, col=2,
    )

fig3.add_vline(x=0,   line_color='#BBBBBB', line_width=1.2, row=1, col=2)
fig3.add_hline(y=3.5, line_dash='dot', line_color='#555', line_width=1.8, row=1, col=2)
fig3.add_annotation(
    xref='x2', yref='y2',
    x=0.25, y='CO2 per capita',
    text='← Small independent<br>effect',
    showarrow=False,
    font=dict(size=10, color=CORAL, family='Arial'),
)
fig3.add_annotation(
    xref='paper', yref='y2',
    x=1.0, y=3.25,
    text='Development<br>indicators',
    showarrow=False,
    font=dict(size=10, color='#555', family='Arial'),
)

fig3.update_xaxes(showgrid=False, zeroline=False,
                  tickfont=dict(size=10), row=1, col=1)
fig3.update_yaxes(showgrid=True, gridcolor='#F0F0F0',
                  range=[0, 1.12], tickfont=dict(size=11),
                  title='R²', row=1, col=1)
fig3.update_xaxes(showgrid=True, gridcolor='#F0F0F0',
                  zeroline=False, tickfont=dict(size=10),
                  title='Standardised β', row=1, col=2,
                  range=[min(coefs) - 0.15, max(coefs) + 0.25])
fig3.update_yaxes(showgrid=False, tickfont=dict(size=11), row=1, col=2,
                  range=[-0.5, len(FEATURE_LABELS) - 0.5])

fig3.update_layout(
    title=dict(
        text='<b>How Much Does CO2 Independently Explain Life Satisfaction?</b>',
        font=dict(size=15, family='Arial', color=SLATE),
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color=SLATE),
    height=540, width=1120,
    margin=dict(l=60, r=80, t=100, b=60),
    barmode='overlay',
    legend=dict(
        title=dict(text='<b>Model</b>', font=dict(size=11)),
        x=1.01, y=1,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#E0E0E0', borderwidth=1,
    ),
)

#### Actual App

In [13]:
from dash import Dash, dcc, html, Input, Output, State, no_update
import json

# trace index constants
#   fw.data[0–3]   left panel bars  (Linear, Quadratic, Cubic, Multiple)
#   fw.data[4–10]  right panel bars (one per feature)
MULTI_IDX  = 3
FEAT_START = 4

# dash app
app = Dash(__name__)

app.layout = html.Div([

    # dcc.Store holds the toggle state as a JSON list of booleans
    dcc.Store(id='active-mask', data=json.dumps([True] * len(FEATURES))),

    dcc.Graph(
        id='fig3',
        figure=copy.deepcopy(fig3),
        config={'displayModeBar': True},
        style={'height': '560px'},
    ),

    html.Div(
        'Click any coefficient bar (right panel) to toggle that feature '
        'on/off. The Multiple Regression R² bar and reference line update live.',
        style={
            'fontFamily': 'Arial', 'fontSize': '12px',
            'color': '#666', 'marginTop': '6px', 'marginLeft': '60px',
        }
    ),
], style={'backgroundColor': 'white', 'padding': '20px'})


@app.callback(
    Output('fig3',        'figure'),
    Output('active-mask', 'data'),
    Input('fig3',         'clickData'),
    State('fig3',         'figure'),
    State('active-mask',  'data'),
    prevent_initial_call=True,
)

def toggle_feature(click_data, current_fig, mask_json):
    if click_data is None:
        return no_update, no_update

    curve_num = click_data['points'][0]['curveNumber']

    # only respond to right-panel feature traces (indices 4–10)
    if curve_num < FEAT_START:
        return no_update, no_update

    feat_idx    = curve_num - FEAT_START
    active_mask = json.loads(mask_json)
    active_mask[feat_idx] = not active_mask[feat_idx]

    # recompute R^2 with active features only
    active_idx = [i for i, on in enumerate(active_mask) if on]
    if active_idx:
        X_sub  = X_scaled[:, active_idx]
        m      = LinearRegression().fit(X_sub, y_all)
        new_r2 = r2_score(y_all, m.predict(X_sub))
    else:
        new_r2 = 0.0

    # patch the figure dict in-place
    fig = copy.deepcopy(current_fig)

    # toggle the clicked coefficient bar colour
    t = fig['data'][curve_num]
    if active_mask[feat_idx]:
        t['marker']['color']   = bar_colors_right[feat_idx]
        t['marker']['opacity'] = 1.0
    else:
        t['marker']['color']   = INACTIVE_COLOR
        t['marker']['opacity'] = 0.45

    # update the Multiple Regression bar
    fig['data'][MULTI_IDX]['y']    = [new_r2]
    fig['data'][MULTI_IDX]['text'] = [f'{new_r2:.2f}']
    fig['data'][MULTI_IDX]['hovertemplate'] = (
        f'<b>Multiple Regression</b><br>R² = {new_r2:.3f}<extra></extra>'
    )

    # move the dashed hline (shapes[0] = the coral hline on the left panel)
    fig['layout']['shapes'][0]['y0'] = new_r2
    fig['layout']['shapes'][0]['y1'] = new_r2

    # update the "Multiple R^2 = X.XX" floating annotation
    for ann in fig['layout']['annotations']:
        if ann.get('text', '').startswith('Multiple R²'):
            ann['text'] = f'Multiple R² = {new_r2:.2f}'
            ann['y']    = new_r2 + 0.05
            break

    return fig, json.dumps(active_mask)


# run
if __name__ == '__main__':
    app.run(debug=True, port=8050)

#app.run(debug=True, port=8050)

#### Depreciated Version (non-interactive)

In [14]:
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots

# colors
TEAL_GRAD = ['#A8D5CF', '#1A9C87', '#155F56']
CORAL = '#E8734A'
TEAL_POS = '#1A9C87'
SLATE = '#2E3B4E'

r2_vals = [r2_lin, r2_quad, r2_cub, r2_multi]
bar_colors_left = TEAL_GRAD + [CORAL]
bar_labels_left = ['Linear (CO2 only)', 'Quadratic (CO2 only)',
                    'Cubic (CO2 only)', 'Multiple Regression']
bar_colors_right = [TEAL_POS if abs(c) > 0.05 else CORAL for c in coefs]

# static fig
fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>R² by Model</b>',
        '<b>Standardised Regression Coefficients</b><br>'
        '<span style="font-size:11px">Multiple Regression (bootstrapped 95% CIs)</span>',
    ],
    column_widths=[0.40, 0.60],
    horizontal_spacing=0.14,
)

# left panel: model bars
for label, val, color in zip(bar_labels_left, r2_vals, bar_colors_left):
    fig3.add_trace(
        go.Bar(
            name=label,
            x=[label],
            y=[val],
            marker_color=color,
            marker_line=dict(color='white', width=1.5),
            text=[f'{val:.2f}'],
            textposition='outside',
            textfont=dict(size=12, color=SLATE),
            width=0.55,
            legendgroup=label,
            hovertemplate=f'<b>{label}</b><br>R² = {val:.3f}<extra></extra>',
        ),
        row=1, col=1,
    )

# Reference dashed line at multiple R^2
fig3.add_hline(
    y=r2_multi, line_dash='dash', line_color=CORAL, line_width=1.8,
    row=1, col=1,
)
fig3.add_annotation(
    xref='x', yref='y',
    x=1.5, y=r2_multi + 0.05,
    text=f'Multiple R² = {r2_multi:.2f}',
    showarrow=False,
    font=dict(size=10, color=CORAL, family='Arial'),
    row=1, col=1,
)

# right panel: coeff bars 
fig3.add_trace(
    go.Bar(
        name='Standardised β',
        x=coefs,
        y=FEATURE_LABELS,
        orientation='h',
        marker_color=bar_colors_right,
        marker_line=dict(color='white', width=0.8),
        error_x=dict(
            type='data', symmetric=False,
            array=ci_hi - coefs,
            arrayminus=coefs - ci_lo,
            color='#666', thickness=1.8, width=5,
        ),
        showlegend=False,
    ),
    row=1, col=2,
)

# right panel: decorators
fig3.add_vline(x=0, line_color='#BBBBBB', line_width=1.2, row=1, col=2)
fig3.add_hline(y=3.5, line_dash='dot', line_color='#555', line_width=1.8, row=1, col=2)

fig3.add_annotation(
    xref='x2', yref='y2',
    x=0.2, y='CO2 per capita',
    text='← Small independent<br>effect',
    showarrow=False,
    font=dict(size=10, color=CORAL, family='Arial'),
)
fig3.add_annotation(
    xref='paper', yref='y2',
    x=1.0, y=3.25,
    text='Development<br>indicators',
    showarrow=False,
    font=dict(size=10, color='#555', family='Arial'),
)

# styling
fig3.update_xaxes(showgrid=False, zeroline=False,
                  tickfont=dict(size=10), row=1, col=1)
fig3.update_yaxes(showgrid=True, gridcolor='#F0F0F0',
                  range=[0, 1.12], tickfont=dict(size=11),
                  title='R²', row=1, col=1)
fig3.update_xaxes(showgrid=True, gridcolor='#F0F0F0',
                  zeroline=False, tickfont=dict(size=10),
                  title='Standardised β', row=1, col=2)
fig3.update_yaxes(showgrid=False, tickfont=dict(size=11), row=1, col=2)

# layout
fig3.update_layout(
    title=dict(
        text='<b>How Much Does CO2 Independently Explain Life Satisfaction?</b>',
        font=dict(size=15, family='Arial', color=SLATE),
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color=SLATE),
    height=540, width=1120,
    margin=dict(l=60, r=80, t=100, b=60),
    barmode='group',
    legend=dict(
        title=dict(text='<b>Model</b>', font=dict(size=11)),
        x=1.01, y=1,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#E0E0E0', borderwidth=1,
    ),
)